In [ ]:
# ---- 0. 环境设置 -----------------------------------------------------------
Sys.setenv(SP_EVOLUTION_STATUS = "2")
set.seed(1)

suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(readr)
  library(stringr)
})

# ---- 1. 参数设置 -----------------------------------------------------------
date_str <- "2026-07-29_1"

embryo_dir <- "/project/work/2026-07-06_development_DEG/Data/"
adult_dir <- "/project/work/2026-06-10_adult_DEG/Data/02_regional_degs/"
data_dir <- "/project/work/2026-07-29_trajectory_class/Data/"

padj_cutoff <- 0.05
logfc_cutoff <- 0.5

stage_order <- c("Early", "Mid", "Late", "Young", "Old")
celltype_order <- c("Rod", "Cone", "AC", "BC", "HC", "MG", "RGC", "RPC")
required_columns <- c("gene", "baseMean", "log2FoldChange", "padj")

dir.create(data_dir, recursive = TRUE, showWarnings = FALSE)

In [ ]:
# ---- 2. 读取五阶段完整 DESeq2 结果 ----------------------------------------
embryo_files <- list.files(
  embryo_dir,
  pattern = "^2026-07-06_2_.*_(Early|Mid|Late)_macula_vs_peripheral\\.csv$",
  full.names = TRUE
)

adult_files <- list.files(
  adult_dir,
  pattern = "^2026-07-28_adult_.*_(Young|Old)_Central_vs_peripheral\\.csv$",
  full.names = TRUE
)

if (length(embryo_files) == 0) {
  stop("未找到胚胎期完整 regional DESeq2 结果：", embryo_dir, call. = FALSE)
}
if (length(adult_files) == 0) {
  stop("未找到 Young/Old 完整 regional DESeq2 结果：", adult_dir, call. = FALSE)
}

embryo_match <- str_match(
  basename(embryo_files),
  "^2026-07-06_2_(.+)_(Early|Mid|Late)_macula_vs_peripheral\\.csv$"
)
adult_match <- str_match(
  basename(adult_files),
  "^2026-07-28_adult_(.+)_(Young|Old)_Central_vs_peripheral\\.csv$"
)

file_info <- bind_rows(
  tibble(
    file = embryo_files,
    celltype = embryo_match[, 2],
    stage = embryo_match[, 3]
  ),
  tibble(
    file = adult_files,
    celltype = adult_match[, 2],
    stage = adult_match[, 3]
  )
)

if (anyNA(file_info$celltype) || anyNA(file_info$stage)) {
  stop("部分输入文件名无法解析 celltype 或 stage。", call. = FALSE)
}

regional_unfiltered <- bind_rows(lapply(seq_len(nrow(file_info)), function(i) {
  one_result <- read_csv(file_info$file[[i]], show_col_types = FALSE)

  if (!all(required_columns %in% colnames(one_result))) {
    stop(
      "输入文件缺少必要列：",
      file_info$file[[i]],
      call. = FALSE
    )
  }

  one_result %>%
    select(all_of(required_columns)) %>%
    mutate(
      celltype = file_info$celltype[[i]],
      stage = file_info$stage[[i]],
      result_available = TRUE,
      .before = 1
    )
}))

if (anyDuplicated(regional_unfiltered[c("celltype", "stage", "gene")]) > 0) {
  stop("完整 DESeq2 结果中存在重复的 celltype × stage × gene。", call. = FALSE)
}

In [ ]:
# ---- 3. 补齐五阶段并定义 M / P / N / NA -----------------------------------
regional_long <- regional_unfiltered %>%
  distinct(celltype, gene) %>%
  crossing(stage = stage_order) %>%
  left_join(
    regional_unfiltered,
    by = c("celltype", "gene", "stage")
  ) %>%
  mutate(
    stage = factor(stage, levels = stage_order),
    test_status = case_when(
      is.na(result_available) ~ "result_absent",
      is.na(padj) | is.na(log2FoldChange) ~ "untested_or_undetermined",
      TRUE ~ "tested"
    ),
    status = case_when(
      test_status != "tested" ~ "NA",
      padj < padj_cutoff & log2FoldChange > logfc_cutoff ~ "M",
      padj < padj_cutoff & log2FoldChange < -logfc_cutoff ~ "P",
      TRUE ~ "N"
    )
  ) %>%
  arrange(match(celltype, celltype_order), celltype, gene, stage)

if (any(regional_long$status == "N" & regional_long$test_status != "tested")) {
  stop("发现未实际检验的阶段被错误标记为 N。", call. = FALSE)
}

regional_wide <- regional_long %>%
  select(
    celltype,
    gene,
    stage,
    status,
    test_status,
    baseMean,
    log2FoldChange,
    padj
  ) %>%
  pivot_wider(
    names_from = stage,
    values_from = c(status, test_status, baseMean, log2FoldChange, padj),
    names_glue = "{stage}_{.value}"
  ) %>%
  mutate(
    Early_Mid_Late_Young_Old = paste(
      Early_status,
      Mid_status,
      Late_status,
      Young_status,
      Old_status,
      sep = "\u2013"
    ),
    classification_group = case_when(
      celltype %in% c("Rod", "MG") ~ "full_from_mid",
      celltype == "BC" ~ "full_from_late",
      celltype %in% c("RGC", "RPC") ~ "embryonic_only",
      TRUE ~ "full_five_stage"
    ),
    classification_scope = if_else(
      classification_group == "embryonic_only",
      "Embryonic only",
      "Full lifecycle"
    ),
    classification_stages = case_when(
      classification_group == "full_from_mid" ~ "Mid;Late;Young;Old",
      classification_group == "full_from_late" ~ "Late;Young;Old",
      classification_group == "embryonic_only" ~ "Early;Mid;Late",
      TRUE ~ "Early;Mid;Late;Young;Old"
    )
  )

In [ ]:
# ---- 4. 建立不同可观察窗口的互斥分类映射 -----------------------------------
# 用途：根据实际参与分类的阶段状态，计算生物学类别及必要的辅助注释。
classify_observed_trajectory <- function(states, observed_stages, classification_scope) {
  active_index <- which(states %in% c("M", "P"))
  active_states <- states[active_index]
  active_stage_count <- length(active_index)

  active_block_count <- if (active_stage_count == 0) {
    0L
  } else {
    1L + sum(diff(active_index) > 1)
  }

  switch_position <- if (active_stage_count <= 1) {
    integer(0)
  } else {
    which(active_states[-1] != active_states[-active_stage_count]) + 1L
  }
  direction_switch_count <- length(switch_position)

  # onset_stage 表示首次观察到区域差异的时期，不等同于真实生物学起始时期。
  onset_stage <- if (active_stage_count == 0) {
    NA_character_
  } else {
    observed_stages[active_index[1]]
  }
  end_stage <- if (active_stage_count == 0) {
    NA_character_
  } else {
    observed_stages[active_index[active_stage_count]]
  }
  first_switch_stage <- if (direction_switch_count == 0) {
    NA_character_
  } else {
    observed_stages[active_index[switch_position[1]]]
  }
  switch_stages <- if (direction_switch_count == 0) {
    NA_character_
  } else {
    paste(observed_stages[active_index[switch_position]], collapse = ";")
  }

  if (classification_scope == "Full lifecycle") {
    Young <- states[observed_stages == "Young"]
    Old <- states[observed_stages == "Old"]

    aging_change <- case_when(
      Young == "NA" | Old == "NA" ~ "undetermined",
      Young == Old & Young == "N" ~ "no_change_neutral",
      Young == Old & Young %in% c("M", "P") ~ "maintained",
      Young %in% c("M", "P") & Old == "N" ~ "loss",
      Young == "N" & Old %in% c("M", "P") ~ "gain",
      Young %in% c("M", "P") & Old %in% c("M", "P") &
        Young != Old ~ "direction_switch",
      TRUE ~ "undetermined"
    )
  } else {
    aging_change <- NA_character_
  }

  if (any(states == "NA")) {
    return(tibble(
      primary_class = "Insufficient evidence",
      subclass = "Untested/undetermined",
      direction_class = "Undetermined",
      onset_stage = NA_character_,
      end_stage = NA_character_,
      active_stage_count = NA_integer_,
      active_block_count = NA_integer_,
      direction_switch_count = NA_integer_,
      first_switch_stage = NA_character_,
      switch_stages = NA_character_,
      aging_change = "undetermined"
    ))
  }

  if (all(states == "N")) {
    return(tibble(
      primary_class = "No regional DEG",
      subclass = "No regional DEG",
      direction_class = "No direction",
      onset_stage = onset_stage,
      end_stage = end_stage,
      active_stage_count = active_stage_count,
      active_block_count = active_block_count,
      direction_switch_count = direction_switch_count,
      first_switch_stage = first_switch_stage,
      switch_stages = switch_stages,
      aging_change = aging_change
    ))
  }

  if (all(c("M", "P") %in% active_states)) {
    direction_class <- case_when(
      direction_switch_count == 1 & active_states[1] == "M" ~ "M-to-P",
      direction_switch_count == 1 & active_states[1] == "P" ~ "P-to-M",
      direction_switch_count >= 2 ~ "Multiple switch",
      TRUE ~ NA_character_
    )

    return(tibble(
      primary_class = "Complex remodeling",
      subclass = "M-P direction switch",
      direction_class = direction_class,
      onset_stage = onset_stage,
      end_stage = end_stage,
      active_stage_count = active_stage_count,
      active_block_count = active_block_count,
      direction_switch_count = direction_switch_count,
      first_switch_stage = first_switch_stage,
      switch_stages = switch_stages,
      aging_change = aging_change
    ))
  }

  direction_class <- if (all(active_states == "M")) {
    "Macular direction"
  } else {
    "Peripheral direction"
  }

  if (active_block_count > 1) {
    return(tibble(
      primary_class = "Complex remodeling",
      subclass = "Intermittent/reactivated",
      direction_class = direction_class,
      onset_stage = onset_stage,
      end_stage = end_stage,
      active_stage_count = active_stage_count,
      active_block_count = active_block_count,
      direction_switch_count = direction_switch_count,
      first_switch_stage = first_switch_stage,
      switch_stages = switch_stages,
      aging_change = aging_change
    ))
  }

  onset_index <- active_index[1]
  end_index <- active_index[active_stage_count]

  if (classification_scope == "Embryonic only") {
    class_result <- case_when(
      end_stage != "Late" ~
        "Embryonic-only trajectory|Embryonic transient",
      onset_stage == "Late" ~
        "Embryonic-only trajectory|Embryonic late-emergent",
      end_stage == "Late" ~
        "Embryonic-only trajectory|Embryonic sustained to Late",
      TRUE ~ NA_character_
    )
  } else {
    class_result <- case_when(
      onset_stage %in% c("Early", "Mid", "Late") &
        end_stage %in% c("Early", "Mid", "Late") ~
        "Developmental-origin program|Developmental transient",
      onset_stage %in% c("Early", "Mid", "Late") & end_stage == "Young" ~
        "Developmental-origin program|Developmental aging-lost",
      onset_stage %in% c("Early", "Mid", "Late") & end_stage == "Old" ~
        "Developmental-origin program|Developmental maintained",
      onset_stage == "Young" & end_stage == "Young" ~
        "Adult-acquired program|Young-specific",
      onset_stage == "Young" & end_stage == "Old" ~
        "Adult-acquired program|Adult acquired and maintained",
      onset_stage == "Old" & end_stage == "Old" ~
        "Old-acquired program|Old-acquired",
      TRUE ~ NA_character_
    )
  }

  tibble(
    primary_class = str_split_fixed(class_result, "\\|", 2)[, 1],
    subclass = str_split_fixed(class_result, "\\|", 2)[, 2],
    direction_class = direction_class,
    onset_stage = onset_stage,
    end_stage = end_stage,
    active_stage_count = active_stage_count,
    active_block_count = active_block_count,
    direction_switch_count = direction_switch_count,
    first_switch_stage = first_switch_stage,
    switch_stages = switch_stages,
    aging_change = aging_change
  )
}

state_levels <- c("M", "P", "N", "NA")

raw_trajectory_grid <- crossing(
  Early_status = state_levels,
  Mid_status = state_levels,
  Late_status = state_levels,
  Young_status = state_levels,
  Old_status = state_levels
)

classification_group_info <- list(
  list(
    classification_group = "full_five_stage",
    classification_scope = "Full lifecycle",
    observed_stages = c("Early", "Mid", "Late", "Young", "Old")
  ),
  list(
    classification_group = "full_from_mid",
    classification_scope = "Full lifecycle",
    observed_stages = c("Mid", "Late", "Young", "Old")
  ),
  list(
    classification_group = "full_from_late",
    classification_scope = "Full lifecycle",
    observed_stages = c("Late", "Young", "Old")
  ),
  list(
    classification_group = "embryonic_only",
    classification_scope = "Embryonic only",
    observed_stages = c("Early", "Mid", "Late")
  )
)

trajectory_map <- bind_rows(lapply(classification_group_info, function(one_group) {
  trajectory_classification <- bind_rows(
    lapply(seq_len(nrow(raw_trajectory_grid)), function(i) {
      observed_states <- unlist(
        raw_trajectory_grid[
          i,
          paste0(one_group$observed_stages, "_status")
        ],
        use.names = FALSE
      )

      classify_observed_trajectory(
        states = observed_states,
        observed_stages = one_group$observed_stages,
        classification_scope = one_group$classification_scope
      )
    })
  )

  bind_cols(raw_trajectory_grid, trajectory_classification) %>%
    mutate(
      classification_group = one_group$classification_group,
      classification_scope = one_group$classification_scope,
      classification_stages = paste(one_group$observed_stages, collapse = ";"),
      classification_trajectory = apply(
        select(., all_of(paste0(one_group$observed_stages, "_status"))),
        1,
        paste,
        collapse = "\u2013"
      ),
      Early_Mid_Late_Young_Old = paste(
        Early_status,
        Mid_status,
        Late_status,
        Young_status,
        Old_status,
        sep = "\u2013"
      )
    )
}))

In [ ]:
# ---- 5. 合并分类并验证 -----------------------------------------------------
if (
  nrow(trajectory_map) != length(classification_group_info) * 4^5 ||
    anyDuplicated(
      trajectory_map[c("classification_group", "Early_Mid_Late_Young_Old")]
    ) > 0 ||
    anyNA(trajectory_map$primary_class) ||
    anyNA(trajectory_map$subclass) ||
    anyNA(trajectory_map$direction_class) ||
    any(str_detect(trajectory_map$primary_class, "Other")) ||
    any(str_detect(trajectory_map$subclass, "Other"))
) {
  stop("不同可观察时间窗口的状态组合未能获得互斥且完整的分类。", call. = FALSE)
}

ppmmn_check <- trajectory_map %>%
  filter(
    classification_group == "full_five_stage",
    Early_Mid_Late_Young_Old == "P\u2013P\u2013M\u2013M\u2013N"
  )

if (
  nrow(ppmmn_check) != 1 ||
    ppmmn_check$primary_class != "Complex remodeling" ||
    ppmmn_check$subclass != "M-P direction switch" ||
    ppmmn_check$direction_class != "P-to-M" ||
    ppmmn_check$first_switch_stage != "Late" ||
    ppmmn_check$aging_change != "loss"
) {
  stop("P-P-M-M-N 的分类或辅助注释不符合预期。", call. = FALSE)
}

rod_mg_check <- trajectory_map %>%
  filter(
    classification_group == "full_from_mid",
    Early_Mid_Late_Young_Old == "NA\u2013M\u2013M\u2013M\u2013M"
  )

bc_check <- trajectory_map %>%
  filter(
    classification_group == "full_from_late",
    Early_Mid_Late_Young_Old == "NA\u2013NA\u2013M\u2013M\u2013N"
  )

embryonic_check <- trajectory_map %>%
  filter(
    classification_group == "embryonic_only",
    Early_Mid_Late_Young_Old == "N\u2013M\u2013M\u2013NA\u2013NA"
  )

rgc_old_check <- trajectory_map %>%
  filter(
    classification_group == "embryonic_only",
    Early_status == "N",
    Mid_status == "N",
    Late_status == "M",
    Young_status == "NA",
    Old_status %in% c("M", "N")
  )

if (
  nrow(rod_mg_check) != 1 ||
    rod_mg_check$subclass != "Developmental maintained" ||
    rod_mg_check$onset_stage != "Mid" ||
    nrow(bc_check) != 1 ||
    bc_check$subclass != "Developmental aging-lost" ||
    bc_check$onset_stage != "Late" ||
    nrow(embryonic_check) != 1 ||
    embryonic_check$primary_class != "Embryonic-only trajectory" ||
    embryonic_check$subclass != "Embryonic sustained to Late" ||
    nrow(rgc_old_check) != 2 ||
    n_distinct(rgc_old_check$primary_class, rgc_old_check$subclass) != 1 ||
    any(!is.na(rgc_old_check$aging_change))
) {
  stop("结构性缺失细胞类型的分类检查失败。", call. = FALSE)
}

trajectory_result <- regional_wide %>%
  left_join(
    trajectory_map %>%
      select(
        classification_group,
        Early_Mid_Late_Young_Old,
        classification_trajectory,
        primary_class,
        subclass,
        direction_class,
        onset_stage,
        end_stage,
        active_block_count,
        direction_switch_count,
        first_switch_stage,
        aging_change
      ),
    by = c("classification_group", "Early_Mid_Late_Young_Old")
  ) %>%
  select(
    celltype,
    gene,
    Early_Mid_Late_Young_Old,
    classification_scope,
    classification_stages,
    classification_trajectory,
    primary_class,
    subclass,
    direction_class,
    onset_stage,
    end_stage,
    active_block_count,
    direction_switch_count,
    first_switch_stage,
    aging_change,
    Early_status,
    Early_test_status,
    Early_baseMean,
    Early_log2FoldChange,
    Early_padj,
    Mid_status,
    Mid_test_status,
    Mid_baseMean,
    Mid_log2FoldChange,
    Mid_padj,
    Late_status,
    Late_test_status,
    Late_baseMean,
    Late_log2FoldChange,
    Late_padj,
    Young_status,
    Young_test_status,
    Young_baseMean,
    Young_log2FoldChange,
    Young_padj,
    Old_status,
    Old_test_status,
    Old_baseMean,
    Old_log2FoldChange,
    Old_padj
  ) %>%
  arrange(match(celltype, celltype_order), celltype, gene)

if (
  anyDuplicated(trajectory_result[c("celltype", "gene")]) > 0 ||
    nrow(trajectory_result) != n_distinct(
      regional_unfiltered$celltype,
      regional_unfiltered$gene
    ) ||
    anyNA(trajectory_result$primary_class)
) {
  stop("最终 celltype × gene 轨迹表的唯一性或完整性检查失败。", call. = FALSE)
}

In [ ]:
# ---- 6. 保存结果 -----------------------------------------------------------
output_file <- file.path(
  data_dir,
  paste0(date_str, "_five_stage_regional_DEG_trajectory_classified.csv")
)

# 输出：按细胞类型可观察时间窗口整理的 regional DEG 状态和互斥轨迹分类主表，供后续作图、富集和候选程序筛选。
write_csv(trajectory_result, output_file, na = "NA")

cat("区域差异轨迹分类完成。\n")
cat("输入文件数：", nrow(file_info), "\n")
cat("输出 celltype × gene 数：", nrow(trajectory_result), "\n")
cat("输出文件：", output_file, "\n")

In [ ]:
trajectory_result <- read.csv('/project/work/2026-07-29_trajectory_class/Data//2026-07-29_1_five_stage_regional_DEG_trajectory_classified.csv')

In [ ]:
trajectory_result$subclass_direction <- paste0(trajectory_result$subclass,'_',trajectory_result$direction_class)